# Healthcare Cost Optimization Analysis – Charité Berlin
**Author:** Kevin  
**Date:** March 2026  
**Data:** Publicly available — Charité Annual Reports, German Hospital Institute, academic benchmarks

---

This notebook supports the case study with lightweight data analysis: visualizing Charité's financial trend, benchmarking against sector averages, and modeling the projected impact of three cost-saving pillars.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#F0F6F8',
    'axes.facecolor': '#FFFFFF',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

TEAL   = '#028090'
SEAFOAM = '#02C39A'
RED    = '#DC2626'
AMBER  = '#D97706'
NAVY   = '#0D2B3E'
GRAY   = '#6B7280'

print('Libraries loaded.')

## 1. Financial Trend (2020–2024)

In [ ]:
df = pd.read_csv('data/charite_financials.csv')
print(df[['year','total_revenue_eur_m','total_expenditure_eur_m','net_result_eur_m','personnel_costs_eur_m']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Charité – Universitätsmedizin Berlin: Financial Overview 2020–2024",
             fontsize=14, fontweight='bold', color=NAVY, y=1.02)

# ── Left: Revenue vs Expenditure ─────────────────────────────
ax1 = axes[0]
x = df['year']
ax1.fill_between(x, df['total_revenue_eur_m'], df['total_expenditure_eur_m'],
                 where=(df['total_expenditure_eur_m'] > df['total_revenue_eur_m']),
                 alpha=0.15, color=RED, label='Deficit area')
ax1.plot(x, df['total_revenue_eur_m'],   color=TEAL,    lw=2.5, marker='o', ms=6, label='Revenue (€M)')
ax1.plot(x, df['total_expenditure_eur_m'], color=RED, lw=2.5, marker='s', ms=6, linestyle='--', label='Expenditure (€M)')
ax1.set_title('Revenue vs. Expenditure', fontweight='bold', color=NAVY)
ax1.set_ylabel('€ Millions')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}M'))
ax1.legend(fontsize=9)
ax1.set_xticks(x)

# ── Right: Net Result (surplus/deficit) ──────────────────────
ax2 = axes[1]
colors = [RED if v < 0 else SEAFOAM for v in df['net_result_eur_m']]
bars = ax2.bar(x, df['net_result_eur_m'], color=colors, width=0.6, zorder=3)
ax2.axhline(0, color=GRAY, lw=1, linestyle='-')
ax2.set_title('Annual Net Result (€M)', fontweight='bold', color=NAVY)
ax2.set_ylabel('€ Millions')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}M'))
ax2.set_xticks(x)
for bar, val in zip(bars, df['net_result_eur_m']):
    offset = -8 if val < 0 else 2
    ax2.text(bar.get_x() + bar.get_width()/2, val + offset,
             f'€{val:.1f}M', ha='center', va='top' if val < 0 else 'bottom',
             fontsize=9, fontweight='bold', color=RED if val < 0 else SEAFOAM)

plt.tight_layout()
plt.savefig('data/fig1_financial_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 1 saved.')

## 2. Personnel Cost Deep-Dive

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Personnel Cost Analysis – The #1 Cost Driver',
             fontsize=14, fontweight='bold', color=NAVY, y=1.02)

# ── Left: Personnel cost trend ────────────────────────────────
ax1 = axes[0]
ax1.bar(df['year'], df['personnel_costs_eur_m'], color=TEAL, width=0.55, zorder=3, label='Personnel Costs')
ax1.bar(df['year'], df['total_expenditure_eur_m'] - df['personnel_costs_eur_m'],
        bottom=df['personnel_costs_eur_m'], color='#CBD5E1', width=0.55, zorder=3, label='Other Costs')
ax1.set_title('Personnel vs. Other Costs (€M)', fontweight='bold', color=NAVY)
ax1.set_ylabel('€ Millions')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}M'))
ax1.set_xticks(df['year'])
ax1.legend(fontsize=9)

# ── Right: Personnel as % of total expenditure ───────────────
ax2 = axes[1]
pct = (df['personnel_costs_eur_m'] / df['total_expenditure_eur_m'] * 100).round(1)
line = ax2.plot(df['year'], pct, color=TEAL, lw=2.5, marker='o', ms=7, zorder=3)
ax2.axhline(62, color=AMBER, lw=1.5, linestyle='--', label='German hospital avg (62%)')
ax2.fill_between(df['year'], pct, 62,
                 where=(pct < 62), alpha=0.12, color=SEAFOAM, label='Below sector avg')
for xi, yi in zip(df['year'], pct):
    ax2.annotate(f'{yi}%', (xi, yi), textcoords='offset points', xytext=(0, 8),
                 ha='center', fontsize=9, fontweight='bold', color=NAVY)
ax2.set_title('Personnel Cost as % of Total Expenditure', fontweight='bold', color=NAVY)
ax2.set_ylabel('% of Total Expenditure')
ax2.set_ylim(50, 70)
ax2.set_xticks(df['year'])
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('data/fig2_personnel_costs.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 2 saved.')

## 3. Savings Potential Modelling

In [ ]:
benchmarks = pd.read_csv('data/german_hospital_benchmarks.csv')

# Savings model
savings = {
    'Workforce\nRedesign': {'low': 40, 'mid': 60, 'high': 80, 'color': TEAL},
    'OR & Equipment\nOptimization': {'low': 15, 'mid': 22, 'high': 30, 'color': SEAFOAM},
    'Supply Chain\nConsolidation': {'low': 10, 'mid': 15, 'high': 20, 'color': AMBER},
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Projected Annual Savings by Strategic Pillar',
             fontsize=14, fontweight='bold', color=NAVY, y=1.02)

# ── Left: Range chart ─────────────────────────────────────────
ax1 = axes[0]
labels = list(savings.keys())
y_pos = np.arange(len(labels))
for i, (label, v) in enumerate(savings.items()):
    ax1.barh(i, v['high'] - v['low'], left=v['low'], height=0.5,
             color=v['color'], alpha=0.25, zorder=2)
    ax1.scatter([v['mid']], [i], color=v['color'], s=120, zorder=4, label=label.replace('\n', ' '))
    ax1.plot([v['low'], v['high']], [i, i], color=v['color'], lw=3, zorder=3)
    ax1.text(v['low'] - 1, i, f'€{v["low"]}M', ha='right', va='center', fontsize=9, color=GRAY)
    ax1.text(v['high'] + 1, i, f'€{v["high"]}M', ha='left', va='center', fontsize=9, color=GRAY)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(labels)
ax1.set_xlabel('Annual Savings (€M)')
ax1.set_title('Savings Range (Low / Mid / High)', fontweight='bold', color=NAVY)
ax1.set_xlim(0, 95)
ax1.axvline(87.4, color=RED, lw=1.5, linestyle='--', label='2024 Deficit (€87.4M)')
ax1.legend(fontsize=8, loc='lower right')

# ── Right: Cumulative 3-year roadmap ─────────────────────────
ax2 = axes[1]
phases = ['Phase 1\n(0-12 mo)', 'Phase 2\n(12-24 mo)', 'Phase 3\n(24-36 mo)']
cumulative_low  = [15, 30, 65]
cumulative_high = [25, 50, 130]
cumulative_mid  = [20, 40, 97]
x = np.arange(len(phases))
ax2.fill_between(x, cumulative_low, cumulative_high, alpha=0.2, color=TEAL, label='Savings range')
ax2.plot(x, cumulative_mid, color=TEAL, lw=2.5, marker='D', ms=8, zorder=4, label='Mid estimate')
ax2.axhline(87.4, color=RED, lw=1.5, linestyle='--', label=f'2024 Deficit €87.4M')
ax2.fill_between(x, cumulative_high, 87.4,
                 where=[v >= 87.4 for v in cumulative_high],
                 alpha=0.1, color=SEAFOAM, label='Break-even zone')
for xi, lo, hi, mi in zip(x, cumulative_low, cumulative_high, cumulative_mid):
    ax2.annotate(f'€{lo}–{hi}M', (xi, mi), textcoords='offset points',
                 xytext=(0, 12), ha='center', fontsize=9, fontweight='bold', color=NAVY)
ax2.set_xticks(x)
ax2.set_xticklabels(phases)
ax2.set_ylabel('Cumulative Savings (€M)')
ax2.set_title('3-Year Cumulative Savings Roadmap', fontweight='bold', color=NAVY)
ax2.legend(fontsize=9)
ax2.set_ylim(0, 155)

plt.tight_layout()
plt.savefig('data/fig3_savings_roadmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 3 saved.')

## 4. Benchmark Comparison: Charité vs. German Sector

In [ ]:
categories = [
    'OR Utilization\n(%)',
    'Staff Direct\nPatient Time (%)',
    'Personnel Cost\nShare (%)',
]
charite_values  = [65, 20, 55]   # Estimated from benchmarks
target_values   = [85, 45, 50]   # Industry-leading targets
sector_avg      = [65, 25, 62]   # German sector average

x = np.arange(len(categories))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 5.5))
fig.patch.set_facecolor('#F0F6F8')
ax.set_facecolor('#FFFFFF')

bars1 = ax.bar(x - width, charite_values,  width, color=NAVY,    label='Charité (estimated)', zorder=3)
bars2 = ax.bar(x,          sector_avg,     width, color=TEAL,    label='German sector avg',   zorder=3, alpha=0.8)
bars3 = ax.bar(x + width,  target_values,  width, color=SEAFOAM, label='Efficiency target',   zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel('Value (%)')
ax.set_title('Charité vs. Sector Benchmarks vs. Efficiency Targets',
             fontweight='bold', color=NAVY, fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(0, 105)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.text(0.98, 0.97, 'Note: Charité values estimated from sector benchmarks and published data',
        transform=ax.transAxes, ha='right', va='top', fontsize=8, color=GRAY, style='italic')

plt.tight_layout()
plt.savefig('data/fig4_benchmarks.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 4 saved.')

## Summary

The data confirms the strategic narrative:

1. **Charité's deficit has improved** from €134.6M (2023) to €87.4M (2024), but structural reform is still needed to reach break-even.
2. **Personnel costs are the dominant lever** — at €1.6B and growing at 7.2% YoY, even a 5% efficiency gain via smarter scheduling and workflow redesign would save ~€80M.
3. **OR utilization and supply chain are secondary but fast levers** — benchmarks suggest achievable targets that would add €25–50M in annual savings.
4. **A combined 3-year strategy could yield €65–130M annually**, fully closing or reversing the current deficit.

---
*Author: Kevin · March 2026 · Data: Charité Annual Reports, DKI, PMC research*